In [1]:
import pandas as pd
import requests
import time
import random
import re
from bs4 import BeautifulSoup

In [ ]:
INPUT_CSV = "data/raw/isbn_seed_1000.csv"
RAW_OUTPUT = "data/raw/bookfinder_amzn_raw.csv"
MIN_DELAY = 1.2
MAX_DELAY = 2.5
TIMEOUT = 20

def clean(text):
    return re.sub(r"\s+", " ", text).strip() if text else None

def build_url(isbn):
    return (
        f"https://www.bookfinder.com/isbn/{isbn}/"
        f"?binding=ANY&condition=ANY&currency=USD&destination=US"
    )

def extract_price(text):
    if not text:
        return None
    m = re.search(r"([0-9]+(?:\.[0-9]{1,2})?)", text.replace(",", ""))
    return float(m.group(1)) if m else None

def get_book_title(soup):
    h1 = soup.find("h1")
    return clean(h1.get_text()) if h1 else None

def get_author_name(soup):
    # Look for author information - commonly in a span or div near the title
    # Try multiple common patterns
    author = None
    
    # Pattern 1: Look for "by" followed by author name
    author_tag = soup.find(string=re.compile(r"by\s+", re.I))
    if author_tag:
        parent = author_tag.parent
        if parent:
            text = clean(parent.get_text())
            m = re.search(r"by\s+(.+?)(?:\s*\||$)", text, re.I)
            if m:
                author = clean(m.group(1))
    
    # Pattern 2: Look for class or id containing "author"
    if not author:
        author_elem = soup.find(class_=re.compile(r"author", re.I))
        if author_elem:
            author = clean(author_elem.get_text())
    
    # Pattern 3: Look for h2 or similar near h1
    if not author:
        h1 = soup.find("h1")
        if h1:
            next_elem = h1.find_next_sibling()
            if next_elem:
                text = clean(next_elem.get_text())
                if text and len(text) < 100:  # Reasonable author name length
                    author = text
    
    return author

def scrape_isbn(session, isbn):
    url = build_url(isbn)
    r = session.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=TIMEOUT)
    if r.status_code != 200:
        return []
    if re.search(r"given id is invalid|invalid isbn", r.text, re.I):
        return []
    
    soup = BeautifulSoup(r.text, "lxml")
    title = get_book_title(soup)
    author = get_author_name(soup)
    
    rows = []
    nodes = soup.find_all(string=re.compile("Edition:", re.I))
    
    for node in nodes:
        block = node.parent
        for _ in range(6):
            if not block:
                break
            t = block.get_text(" ", strip=True)
            if "Condition:" in t and ("$" in t or "US$" in t):
                break
            block = block.parent
        
        if not block:
            continue
        
        text = clean(block.get_text(" ", strip=True))
        
        # Extract website name
        website = None
        img = block.find("img")
        if img and img.get("alt"):
            website = clean(img["alt"])
        
        # Only process Amazon.com listings
        if not website or "amazon.com" not in website.lower():
            continue
        
        price = extract_price(text)
        edition = None
        m = re.search(r"Edition:\s*([^|]+)", text)
        if m:
            edition = clean(m.group(1))
        
        condition = None
        m = re.search(r"Condition:\s*([^|]+)", text)
        if m:
            condition = clean(m.group(1))
        
        offer_type = "used"
        if condition and "new" in condition.lower():
            offer_type = "new"
        
        rows.append({
            "isbn": isbn,
            "book_title": title,
            "author": author,
            "raw_website_name": website,
            "offer_type": offer_type,
            "price": price,
            "edition": edition,
            "condition": condition,
        })
    
    return rows

In [3]:

def main():
    df = pd.read_csv(INPUT_CSV)
    isbns = df["isbn"].astype(str).tolist()

    all_rows = []

    with requests.Session() as session:
        for i, isbn in enumerate(isbns, 1):
            print(f"[{i}/{len(isbns)}] {isbn}")
            try:
                all_rows.extend(scrape_isbn(session, isbn))
            except Exception as e:
                print("Error:", isbn, e)

            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    out = pd.DataFrame(all_rows)
    out.to_csv(RAW_OUTPUT, index=False)
    print(f"\nSaved raw data → {RAW_OUTPUT}")


if __name__ == "__main__":
    main()


[1/1000] 9780486227818
[2/1000] 9780099537878
[3/1000] 9781411435049
[4/1000] 9781169232495
[5/1000] 9781072826040
[6/1000] 9780606307925
[7/1000] 9780448448862
[8/1000] 9798352499320
[9/1000] 9780394747095
[10/1000] 9798591487577
[11/1000] 9781789430905
[12/1000] 9780140364545
[13/1000] 9780140059052
[14/1000] 9780425195208
[15/1000] 9781015437890
[16/1000] 9781096133315
[17/1000] 9788310109392
[18/1000] 9798650207900
[19/1000] 9780123120489
[20/1000] 9798475024089
[21/1000] 9780833593535
[22/1000] 9789998849679
[23/1000] 9781781805800
[24/1000] 9781019494950
[25/1000] 9781975307196
[26/1000] 9798584492113
[27/1000] 9781415588994
[28/1000] 9780008195649
[29/1000] 9780712610469
[30/1000] 9781790978137
[31/1000] 9781529017205
[32/1000] 9781414297453
[33/1000] 9780688154981
[34/1000] 9780965593199
[35/1000] 9781520772998
[36/1000] 9780448170534
[37/1000] 9780425033845
[38/1000] 9781604242416
[39/1000] 9780439261395
[40/1000] 9781548781286
[41/1000] 9781534871304
[42/1000] 9798657368932
[